In [ ]:
!pip install transformers

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [ ]:
# Step 2: Import Libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd

In [ ]:
import numpy as np
from tqdm import tqdm
from google.colab import files

In [ ]:

# Step 3: Load base DistilGPT2
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
print(" Model loaded successfully!")

✅ Model loaded successfully!


In [ ]:
import pandas as pd

print("\n=== Load Training Dataset from Path ===")
# Specify your local path or mounted drive path here
train_path = '/content/mmlu_subset_2k_renamed.csv'  # <-- Change this path

train_df = pd.read_csv(train_path)
print(f"Training samples: {len(train_df)}")


=== Load Training Dataset from Path ===
Training samples: 2000


In [ ]:
# Corrected Preprocessing
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)
    labels = [-100] * len(inputs.input_ids)  # Ignore all tokens
    label_token_id = tokenizer(label, add_special_tokens=False)['input_ids'][0]
    labels[-1] = label_token_id  # Only predict the last token (A/B/C/D)

    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': labels
    }

train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_distilgpt2",
    num_train_epochs=0.3,    # not full 1 epoch, only partial
    per_device_train_batch_size=2,  # smaller batch size
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,    # log less often
    save_strategy="no",
    #evaluation_strategy="no",
    report_to="none",
    fp16=True  # if your GPU supports it (NVIDIA T4 supports)
)


In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

<ipython-input-23-8a3bc22b4849>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("\n=== Fine-tuning Started ===")
trainer.train()

# Save Fine-tuned Model
model.save_pretrained("./fine_tuned_distilgpt2")
tokenizer.save_pretrained("./fine_tuned_distilgpt2")

print(" Fine-tuning completed and model saved!")


=== Fine-tuning Started ===


Step,Training Loss
50,2.240300
100,2.288700
150,1.689600
200,1.800100
250,1.742600
300,1.699600


✅ Fine-tuning completed and model saved!


In [ ]:
import pandas as pd

print("\n=== Load Test Dataset from Path ===")
test_dataset_path = "/content/human_ranked.csv"  # <-- put your file path or direct link here

test_df = pd.read_csv(test_dataset_path)
print(f"Test samples: {len(test_df)}")



=== Load Test Dataset from Path ===
Test samples: 480


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_distilgpt2")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_distilgpt2")
model.eval()
model = model.to(device)

In [ ]:
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i,c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]
    choice_logits = []

    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())

    return choice_logits

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = -1

    wrong_options = []
    wrong_logits = []
    for i in range(4):
        if i != predicted_choice_idx:
            wrong_options.append(chr(65 + i))
            wrong_logits.append(logits[i])
    distractor_triplet = [x for _, x in sorted(zip(wrong_logits, wrong_options), reverse=True)]

    results.append({
        'question': question,
        'model_name': "distilgpt2",
        'variant': "low",
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3],
        'predicted_triplet': distractor_triplet
    })


100%|██████████| 480/480 [02:25<00:00,  3.31it/s]


In [ ]:
# Save Predictions
final_df = pd.DataFrame(results)
output_file = "/content/predictions_low_group_distilgpt2.csv"
final_df.to_csv(output_file, index=False)

print(f" Predictions saved to {output_file}!")

files.download(output_file)

✅ Predictions saved to /content/predictions_low_group_distilgpt2.csv!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# -----------------------------------------
#  Reload Training Dataset for Variant 2
# -----------------------------------------
train_df = pd.read_csv("/content/mmlu_subset_2k_renamed.csv")


In [ ]:
#  FINE-TUNE VARIANT 2 (Higher settings)
# -----------------------------------------

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

train_dataset_variant2 = Dataset.from_pandas(train_df)
train_dataset_variant2 = train_dataset_variant2.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
#  Reload tokenizer and base model fresh
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:

def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)
    labels = [-100] * len(inputs.input_ids)
    label_token_id = tokenizer(label, add_special_tokens=False)['input_ids'][0]
    labels[-1] = label_token_id

    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': labels
    }

In [ ]:
train_dataset_variant2 = Dataset.from_pandas(train_df)
train_dataset_variant2 = train_dataset_variant2.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args_variant2 = TrainingArguments(
    output_dir="./fine_tuned_distilgpt2_variant2",
    num_train_epochs=0.5,            # longer training
    per_device_train_batch_size=4,   # bigger batch
    learning_rate=5e-5,              # higher learning rate
    weight_decay=0.01,
    logging_dir="./logs_variant2",
    logging_steps=50,
    save_strategy="no",
   # evaluation_strategy="no",
    report_to="none",
    fp16=True
)

In [ ]:
#  Trainer setup
trainer_variant2 = Trainer(
    model=model,
    args=training_args_variant2,
    train_dataset=train_dataset_variant2,
    tokenizer=tokenizer
)


<ipython-input-46-2d37978594ff>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_variant2 = Trainer(


In [ ]:
#  Start training
print("\n=== Fine-tuning VARIANT 2 (medium) Started ===")
trainer_variant2.train()


=== Fine-tuning VARIANT 2 (medium) Started ===


Step,Training Loss
50,2.183800
100,1.526400
150,1.445100
200,1.462100
250,1.445500


TrainOutput(global_step=250, training_loss=1.6125754699707031, metrics={'train_runtime': 2651.3216, 'train_samples_per_second': 0.377, 'train_steps_per_second': 0.094, 'total_flos': 65324187648000.0, 'train_loss': 1.6125754699707031, 'epoch': 0.5})

In [ ]:
#  Save model
model.save_pretrained("./fine_tuned_distilgpt2_variant2")
tokenizer.save_pretrained("./fine_tuned_distilgpt2_variant2")
print(" Fine-tuning VARIANT 2 complete!")

✅ Fine-tuning VARIANT 2 complete!


In [ ]:
print("\n=== Predicting with VARIANT 2 ===")

#  Reload Test Dataset from file
test_df = pd.read_csv("/content/human_ranked.csv")
print(f"Test samples: {len(test_df)}")



=== Predicting with VARIANT 2 ===
Test samples: 480


In [ ]:
#  Reload fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_distilgpt2_variant2")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_distilgpt2_variant2")
model.eval()
model = model.to(device)

In [ ]:
# Prediction helper
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i,c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
# Confirm test_df is correct
print(test_df.columns)


Index(['subject', 'question_id', 'question', 'correct_answer', 'option_0',
       'option_1', 'option_2', 'option_3',
       'distractor_ranking_best_to_worst_Annotator_1',
       'distractor_ranking_best_to_worst_Annotator_2'],
      dtype='object')


In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))

    # Optional: if you want to set correct choice index from test file (only if available)
    if 'correct_answer' in row:
        correct_choice_index = int(row['correct_answer'])
    else:
        correct_choice_index = -1

    results.append({
        'question': question,
        'model_name': "distilgpt2",             #  Set model name
        'variant': "medium",                     #  Set variant
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': row['option_0'],
        'option_1': row['option_1'],
        'option_2': row['option_2'],
        'option_3': row['option_3']
    })

# Create final DataFrame
final_predictions_df = pd.DataFrame(results)

# Save to CSV
output_path = "/content/predictions_final_format.csv"
final_predictions_df.to_csv(output_path, index=False)

print(f" Predictions saved correctly at {output_path}")


100%|██████████| 480/480 [02:14<00:00,  3.57it/s]

✅ Predictions saved correctly at /content/predictions_final_format.csv


In [ ]:
#  Reload training file again
train_df = pd.read_csv("/content/mmlu_subset_2k_renamed.csv")
print(f"Training samples reloaded for Variant 3: {len(train_df)}")

Training samples reloaded for Variant 3: 2000


In [ ]:
#  Reload model and tokenizer fresh
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
# Preprocessing function
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['option_0']}\nB. {example['option_1']}\nC. {example['option_2']}\nD. {example['option_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['correct_answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)
    labels = [-100] * len(inputs.input_ids)
    label_token_id = tokenizer(label, add_special_tokens=False)['input_ids'][0]
    labels[-1] = label_token_id

    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': labels
    }


In [ ]:
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]   #  train.csv has "answer"

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)
    labels = [-100] * len(inputs.input_ids)
    label_token_id = tokenizer(label, add_special_tokens=False)['input_ids'][0]
    labels[-1] = label_token_id

    return {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': labels
    }


In [ ]:
train_dataset_variant3 = Dataset.from_pandas(train_df)
train_dataset_variant3 = train_dataset_variant3.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
#  Faster Training Arguments for Variant 3
training_args_variant3 = TrainingArguments(
    output_dir="./fine_tuned_distilgpt2_variant3",
    num_train_epochs=0.2,             #  20% of full epoch (instead of 70%)
    per_device_train_batch_size=4,     # Reduce batch size to 4
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_dir="./logs_variant3",
    logging_steps=100,
    save_strategy="no",
    #evaluation_strategy="no",          # No eval during training (only train)
    report_to="none",
    fp16=True
)


In [ ]:
trainer_variant3 = Trainer(
    model=model,
    args=training_args_variant3,
    train_dataset=train_dataset_variant3,
    tokenizer=tokenizer
)

<ipython-input-67-db622f90b622>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_variant3 = Trainer(


In [ ]:
print("\n=== Fine-tuning VARIANT 3 (high) Started ===")
trainer_variant3.train()


=== Fine-tuning VARIANT 3 (high) Started ===


Step,Training Loss
100,1.907400


TrainOutput(global_step=100, training_loss=1.907418212890625, metrics={'train_runtime': 1067.6091, 'train_samples_per_second': 0.375, 'train_steps_per_second': 0.094, 'total_flos': 26129675059200.0, 'train_loss': 1.907418212890625, 'epoch': 0.2})

In [ ]:
#  Save model
model.save_pretrained("./fine_tuned_distilgpt2_variant3")
tokenizer.save_pretrained("./fine_tuned_distilgpt2_variant3")
print("✅ Fine-tuning VARIANT 3 complete!")

✅ Fine-tuning VARIANT 3 complete!


In [ ]:
print("\n=== Predicting with VARIANT 3 ===")

#  Reload test.csv before prediction
test_df = pd.read_csv("/content/human_ranked.csv")
print(f"Test samples reloaded: {len(test_df)}")


=== Predicting with VARIANT 3 ===
Test samples reloaded: 480


In [ ]:
# Reload model from saved Variant 3
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_distilgpt2_variant3")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_distilgpt2_variant3")
model.eval()
model = model.to(device)

In [ ]:
# Helper to get logits
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
# Prediction
results_variant3 = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))

    if 'correct_answer' in row:
        correct_choice_index = int(row['correct_answer'])
    else:
        correct_choice_index = -1

    results_variant3.append({
        'question': question,
        'model_name': "distilgpt2",
        'variant': "high",
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': row['option_0'],
        'option_1': row['option_1'],
        'option_2': row['option_2'],
        'option_3': row['option_3']
    })


100%|██████████| 480/480 [02:18<00:00,  3.47it/s]


In [ ]:
#  Save final predictions
final_df_variant3 = pd.DataFrame(results_variant3)
final_df_variant3.to_csv("/content/predictions_low_group_distilgpt2_variant3.csv", index=False)

print(" Predictions Variant 3 saved at /content/predictions_low_group_distilgpt2_variant3.csv")

✅ Predictions Variant 3 saved at /content/predictions_low_group_distilgpt2_variant3.csv
